<a href="https://colab.research.google.com/github/mohammedwayiz18-jpg/GenerativeAI/blob/main/Gan_exp03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
data = torchvision.datasets.FashionMNIST(
    "./data", train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
)
loader = torch.utils.data.DataLoader(data, 128, shuffle=True)

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 270kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.08MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.7MB/s]


In [ ]:
G = nn.Sequential(
    nn.Linear(100, 256), nn.ReLU(),
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 784), nn.Tanh()
).to(device)

In [ ]:
D = nn.Sequential(
    nn.Linear(784, 512), nn.LeakyReLU(0.2),
    nn.Linear(512, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 1), nn.Sigmoid()
).to(device)

In [ ]:
loss = nn.BCELoss()
g_opt = torch.optim.Adam(G.parameters(), lr=0.0002)
d_opt = torch.optim.Adam(D.parameters(), lr=0.0002)

In [ ]:
for epoch in range(15):
    for real, _ in loader:

        real = real.view(-1, 784).to(device)
        n = real.size(0)

        # Discriminator
        noise = torch.randn(n, 100).to(device)
        fake = G(noise)

        d_loss = loss(D(real), torch.ones(n,1).to(device))
        d_loss += loss(D(fake.detach()), torch.zeros(n,1).to(device))

        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        # Generator
        g_loss = loss(D(fake), torch.ones(n,1).to(device))

        g_opt.zero_grad()
        g_loss.backward()
        g_opt.step()

    print(epoch+1, g_loss.item(), d_loss.item())

1 5.654808521270752 0.19317489862442017
2 7.596228122711182 0.0865560993552208
3 4.608307838439941 0.08940175175666809
4 5.468176364898682 0.1417698860168457
5 4.978993892669678 0.22058025002479553
6 5.3141679763793945 0.1962391436100006
7 5.141727447509766 0.20771107077598572
8 6.1533355712890625 0.11576785147190094
9 4.446634292602539 0.21200482547283173
10 3.5410118103027344 0.19484885036945343
11 6.8601508140563965 0.09477270394563675
12 4.833294868469238 0.31214049458503723
13 3.7088356018066406 0.3522408902645111
14 4.336902618408203 0.2684401571750641
15 3.3567821979522705 0.25430381298065186


In [ ]:
fake = G(torch.randn(16,100).to(device))
fake = fake.view(-1,1,28,28).detach().cpu()

In [ ]:
plt.figure(figsize=(6,6))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(fake[i,0], cmap="gray")
    plt.axis("off")
plt.show()